In [227]:
import torch

In [228]:
N_samples = 10
n = (2,3,4)

A_0 = torch.randn(n[0], n[0], device='cuda', dtype=torch.float64)
A_0 = torch.matmul(A_0, A_0.transpose(-1,-2))  # make positive definite
A_1 = torch.randn(n[1], n[1], device='cuda', dtype=torch.float64)
A_1 = torch.matmul(A_1, A_1.transpose(-1,-2))  # make positive definite
A_2 = torch.randn(n[2], n[2], device='cuda', dtype=torch.float64)    
A_2 = torch.matmul(A_2, A_2.transpose(-1,-2))  # make positive definite

A_i = (A_0, A_1, A_2)

In [229]:
L_i = tuple(torch.linalg.cholesky(A) for A in A_i)

In [230]:
A = A_0
for i in range(1, len(A_i)):
    A = torch.kron(A, A_i[i])
A.shape

torch.Size([24, 24])

In [231]:
b = torch.randn(N_samples, *n, device='cuda', dtype=torch.float64)

In [232]:
b.shape

torch.Size([10, 2, 3, 4])

In [233]:
L = torch.kron(torch.kron(L_i[0], L_i[1]), L_i[2])

In [234]:
x_1 = torch.linalg.solve_triangular(L, b.reshape(N_samples, -1), upper=False, left=False)

In [235]:
x_1.shape

torch.Size([10, 24])

In [251]:
def batch_kron_solve_triangular(L_i, b, upper, left=False):
    x = b
    for i in range(len(L_i)):
        x = x.moveaxis(1,-1)
        x = torch.linalg.solve_triangular(L_i[i], x, upper=upper, left=left)
    return x

In [252]:
x_2 = batch_kron_solve_triangular(L_i, b, upper=False)
x_2.shape

torch.Size([10, 2, 3, 4])

In [253]:
x_1 - x_2.reshape(N_samples, -1)

tensor([[ 4.4409e-16,  6.6613e-16, -5.5511e-17,  0.0000e+00,  4.1633e-16,
          4.4409e-16,  1.1102e-16,  0.0000e+00,  0.0000e+00, -4.4409e-16,
          0.0000e+00, -1.0408e-17,  3.3307e-16,  8.8818e-16, -2.2204e-16,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  2.7756e-17,  0.0000e+00,
          6.6613e-16,  4.4409e-16,  0.0000e+00, -2.7756e-17],
        [-1.5959e-16, -3.3307e-16,  0.0000e+00,  1.7521e-16,  4.4409e-16,
          2.2204e-16,  0.0000e+00,  1.1102e-16,  1.1102e-16, -1.1102e-16,
         -1.8041e-16,  6.9389e-18,  0.0000e+00, -2.2204e-16, -8.3267e-17,
          5.5511e-17,  2.2204e-16,  0.0000e+00, -1.1102e-16,  0.0000e+00,
          2.2204e-16,  0.0000e+00, -1.1102e-16,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00, -1.1102e-16,  5.5511e-17,  0.0000e+00,
          0.0000e+00, -1.1102e-16,  0.0000e+00, -3.3307e-16, -4.4409e-16,
          1.1102e-16,  0.0000e+00,  1.6653e-16,  2.2204e-16, -5.5511e-17,
          0.0000e+00,  5.5511e-17,  1.1102e-16,  0.0000e+00,  